In [2]:
!nvidia-smi

Tue Mar 24 14:22:30 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   43C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
!pip install ultralytics roboflow opencv-python-headless -q
!pip install google-adk gcsfs

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 18.9 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.8/95.8 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 37.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 60.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 93.9 MB/s eta 0:00:00ta 0:00:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.25.1 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2026.2.0 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 5.1 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 304.4/304.4 k

In [4]:
import ultralytics
ultralytics.checks()

from ultralytics import YOLO
import cv2
import os
from roboflow import Roboflow

print(" All dependencies installed successfully!")

Ultralytics 8.4.26  Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Setup complete  (4 CPUs, 31.4 GB RAM, 6842.1/8062.4 GB disk)
 All dependencies installed successfully!


In [5]:
import os
import urllib.request

# === Download YOLOv12n Detection Model ===
detection_url = "https://github.com/sunsmarterjie/yolov12/releases/download/v1.0/yolov12n.pt"
detection_path = "yolov12n.pt"

if not os.path.exists(detection_path):
    print(f" Downloading YOLOv12n detection weights...")
    urllib.request.urlretrieve(detection_url, detection_path)
    print(" Detection weights downloaded successfully!")
else:
    print(f" {detection_path} already exists.")

print("\n Model Setup:")
print(f"   Detection: YOLOv12n (custom download)")
print(f"   Classification: YOLOv8n-cls (auto-downloads on first use)")
print("   Both models are now ready for training!")

 yolov12n.pt already exists.

 Model Setup:
   Detection: YOLOv12n (custom download)
   Classification: YOLOv8n-cls (auto-downloads on first use)
   Both models are now ready for training!


In [6]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="rbyJe304ov9xZbafHDYG")
project = rf.workspace("usefulmech").project("corrosion-detection-cckc1")
version = project.version(1)
dataset = version.download("yolov8")

loading Roboflow workspace...
loading Roboflow project...


In [7]:
# Cell 6: Train detection model
from ultralytics import YOLO

# Load pretrained YOLOv12n
model_detect = YOLO("yolov12n.pt")

# Training configuration
results_detect = model_detect.train(
    data=f"{dataset.location}/data.yaml",  # Path to dataset YAML
    epochs=200,                    # Training epochs
    imgsz=640,                     # Image size
    batch=16,                      # Batch size (adjust based on GPU memory)
    device=0,                      # GPU device (0 for first GPU)

    # Hyperparameters (from your methodology)
    optimizer='SGD',               # Stochastic Gradient Descent
    lr0=0.01,                      # Initial learning rate
    lrf=0.01,                      # Final learning rate (cosine annealing)
    momentum=0.937,                # SGD momentum
    weight_decay=0.0005,           # Weight decay

    # Augmentation
    hsv_h=0.015,                   # HSV hue augmentation
    hsv_s=0.7,                     # HSV saturation
    hsv_v=0.4,                     # HSV value
    degrees=15,                    # Rotation augmentation
    flipud=0.0,                    # Vertical flip probability
    fliplr=0.5,                    # Horizontal flip probability
    mosaic=1.0,                    # Mosaic augmentation

    # Other settings
    patience=50,                   # Early stopping patience
    save=True,                     # Save checkpoints
    save_period=10,                # Save every 10 epochs
    project='runs/detect',         # Output directory
    name='corrosion_detection',    # Experiment name
    exist_ok=True,                 # Overwrite existing
    verbose=True                   # Print training progress
)


Ultralytics 8.4.26  Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/Corrosion-Detection-1/data.yaml, degrees=15, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=200, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov12n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=corrosion_detection, nbs=64, nms=False, opset=None, optimize=False, optimizer=SGD, overlap_mask=True

In [13]:
from IPython.display import FileLink

# This generates a clickable link to download your trained model to your PC
FileLink(r'runs/detect/runs/detect/corrosion_detection/weights/best.pt')

/kaggle/working/runs/detect/runs/detect/corrosion_detection/weights/best.pt

In [ ]:
import matplotlib.pyplot as plt
import cv2
import os

# The exact path you found
image_path = "/kaggle/working/runs/detect/runs/detect/corrosion_detection/val_batch0_pred.jpg"

if os.path.exists(image_path):
    print("Image found! Loading results...")
    img = cv2.imread(image_path)
    
    # Make the plot large so you can see the details and labels clearly
    plt.figure(figsize=(20, 20)) 
    plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.title("Your Model's Predictions on Your Dataset")
    plt.axis('off')
    plt.show()
else:
    print("Oops, couldn't find the image at that exact path. Double check for typos!")